# 🎬 Spot Republic · Promo TikTok v3 (estilo davecast + 3D dorado + TV vieja + neón + Spider-Verse)

Hace un TikTok vertical que explica tu república de Discord con **tu voz**:

- **Subtítulos amarillos con brillo**, con la palabra clave en **GRANDE** (estilo davecast).
- **Pantallas negras** con frases del editor que enganchan ("LUEGO TE EXPLICO POR QUÉ").
- **B-roll** de tu Discord (`Server/broll`) a pantalla completa, en **polaroid** o dentro de una **TV vieja**.
- **Títulos 3D dorados** en el fondo, detrás de ti (gancho, capítulos y cierre).
- **Trazos de luz neón** y **momentos Spider-Verse** (cómic con puntos de imprenta, tinta y colores desfasados).
- **Color de cine cálido**, cambio de encuadre cada ~2.5 s, whooshes y efectos de sonido.
- **Música:** lista de fondo en orden con fundidos suaves y canciones a pedido ("editor, pon la de Colombia") que entran suave.

**Antes:** sube tus clips nuevos a `Server/`, el b-roll a `Server/broll` (grabaciones de pantalla del Discord: canales, roles, elecciones, cómo entrar) y la música a `Server/musica`.
**Cómo usarlo:** Entorno de ejecución → Cambiar tipo → **H100** → **Ejecutar todo**.
Resultado en `Server/Edicion_IA/`. Para cambiar algo, edita `PLAN` (celda 5) y corre las celdas 6–9.

In [ ]:
#@title 1 · Instalar herramientas (Whisper, YOLO, vLLM) y montar Google Drive
#@markdown Usa **Entorno de ejecución → Cambiar tipo → GPU H100 (o A100 80 GB)**. Tarda ~3-5 min.
import subprocess, sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'faster-whisper', 'ultralytics', 'anthropic>=0.70'], check=True)
    # vLLM va en un entorno aparte para no romper el torch de Colab (usado por CLIP/YOLO/Whisper)
    if not os.path.exists('/content/vllm_env/bin/vllm'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.12', '/content/vllm_env'], check=True)
        subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/vllm_env/bin/python', '-U', 'vllm'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openai'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')
try:
    print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout or 'Sin GPU (irá más lento)')
except FileNotFoundError:
    print('Sin GPU (irá más lento)')

In [ ]:
#@title 2 · Configuración (estilo davecast + 3D dorado + TV vieja + neón + Spider-Verse)
import os, glob, json, math, random, re, shutil, subprocess, base64, gc, time
import numpy as np, cv2

DRIVE_FOLDER   = "/content/drive/MyDrive/Server"  #@param {type:"string"}
NOMBRE_SERVER  = "Spot Republic"  #@param {type:"string"}
#@markdown Qué es el server (la IA lo usa para entender lo que dices y escribir las notas).
DESCRIPCION_SERVER = "Servidor de Discord que funciona como una república: gobierno, presidente, elecciones, ciudadanos, roles y economía, como un país (tipo México)."  #@param {type:"string"}
DURACION_OBJETIVO_S = 120  #@param {type:"slider", min:45, max:600, step:15}
#@markdown **Clips:** los videos donde hablas van en `Server/` (como siempre). **B-roll:** grabaciones de pantalla del Discord, capturas, fotos → `Server/broll` (el nombre del archivo ayuda a la IA: `canales.mp4`, `elecciones.png`, `como_entrar.mp4`).
CARPETA_BROLL  = "broll"  #@param {type:"string"}
#@markdown **Efectos**
TITULOS_3D_DORADOS = True  #@param {type:"boolean"}
TV_VIEJA       = True  #@param {type:"boolean"}
NEON           = True  #@param {type:"boolean"}
SPIDERVERSE    = True  #@param {type:"boolean"}
#@markdown **Música** (archivos en `Server/musica`). Fondo en orden, separados por coma; vacío = sin música de fondo.
MUSICA_FONDO   = "kwip_flip, caffeine_rush"  #@param {type:"string"}
VOLUMEN_MUSICA = 0.10  #@param {type:"slider", min:0, max:0.5, step:0.01}
#@markdown Canciones a pedido ("editor, pon la de…"): `usar_archivos` las mete desde `Server/musica`; `marcar_para_tiktok` deja el hueco y la lista con el minuto.
MODO_CANCIONES = "usar_archivos"  #@param ["usar_archivos", "marcar_para_tiktok"]
SEGUNDOS_CANCION_PEDIDA = 10  #@param {type:"slider", min:4, max:20, step:1}
VOLUMEN_CANCIONES = 0.8  #@param {type:"slider", min:0.2, max:1, step:0.05}
EFECTOS_SONIDO = True  #@param {type:"boolean"}
CARPETA_MUSICA = "musica"  #@param {type:"string"}
CREDITO_MUSICA = ""  #@param {type:"string"}
#@markdown Texto final (Discord, "link en la bio"…). Vacío = lo propone la IA.
TEXTO_CTA      = ""  #@param {type:"string"}
IDIOMA         = "es"  #@param ["es", "en"]
#@markdown **Motor de IA del editor** · `local_qwen` = Qwen3.8-27B en tu GPU · `claude` = API de Anthropic · `simple` = sin IA.
MOTOR_IA       = "local_qwen"  #@param ["local_qwen", "claude", "simple"]
MODELO_LOCAL   = "Qwen/Qwen3.8-27B"  #@param {type:"string"}
FPS = 30
OUT_W, OUT_H = 1080, 1920
random.seed(7); np.random.seed(7)

if not os.path.isdir(DRIVE_FOLDER):
    raise FileNotFoundError(f"No existe {DRIVE_FOLDER}. Ajusta DRIVE_FOLDER.")
FOLDER = DRIVE_FOLDER
WORK = os.environ.get("SPOT_WORK", "/content/work_v3")
OUT_DIR = os.path.join(FOLDER, "Edicion_IA")
for d in (WORK, f"{WORK}/proxy", f"{WORK}/frames", f"{WORK}/audio", f"{WORK}/broll", OUT_DIR):
    os.makedirs(d, exist_ok=True)
AUDIO_EXT = ('.mp3', '.m4a', '.wav', '.aac', '.ogg', '.flac')
MUSIC_DIR = os.path.join(FOLDER, CARPETA_MUSICA)
SONGS = sorted(f for f in glob.glob(os.path.join(MUSIC_DIR, '*')) if f.lower().endswith(AUDIO_EXT + ('.mp4',)))
BROLL_DIR = os.path.join(FOLDER, CARPETA_BROLL)
BROLL_FILES = sorted(f for f in glob.glob(os.path.join(BROLL_DIR, '*'))
                     if f.lower().endswith(('.mp4', '.mov', '.webm', '.mkv', '.png', '.jpg', '.jpeg', '.webp')))
print("Carpeta:", FOLDER, "\nSalida:", OUT_DIR)
print(f"Canciones ({len(SONGS)}):", ', '.join(os.path.basename(f) for f in SONGS) or '—')
print(f"B-roll ({len(BROLL_FILES)}):", ', '.join(os.path.basename(f) for f in BROLL_FILES) or f"ninguno — súbelo a {BROLL_DIR}")

In [ ]:
#@title 3 · Inventario, quitar duplicados y crear proxies (hasta 1440p para el recorte vertical)
def run(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        raise RuntimeError(f"Falló: {' '.join(map(str, cmd))[:300]}\n{r.stderr[-1500:]}")
    return r

def probe(path):
    r = run(['ffprobe', '-v', 'error', '-print_format', 'json', '-show_streams', '-show_format', path])
    j = json.loads(r.stdout)
    v = next((s for s in j['streams'] if s['codec_type'] == 'video'), None)
    a = next((s for s in j['streams'] if s['codec_type'] == 'audio'), None)
    return {'duration': float(j['format'].get('duration', 0)), 'video': v, 'audio': a}

def pick_encoder():
    try:
        run(['ffmpeg', '-v', 'error', '-f', 'lavfi', '-i', 'color=s=256x256:d=0.2', '-c:v', 'h264_nvenc', '-f', 'null', '-'])
        return ['-c:v', 'h264_nvenc', '-preset', 'p6', '-cq', '19', '-b:v', '0']
    except Exception:
        return ['-c:v', 'libx264', '-preset', 'medium', '-crf', '18']
VENC = pick_encoder()
print("Encoder:", VENC[1])

all_mp4 = sorted(glob.glob(os.path.join(FOLDER, '*.mp4')), key=lambda p: (re.sub(r' \(\d+\)(?=\.mp4$)', '', p), bool(re.search(r' \(\d+\)\.mp4$', p)), p))  # originales antes que 'nombre (1).mp4'
clips, seen = [], set()
for p in all_mp4:
    name = os.path.basename(p)
    if name.startswith('spot_republic') or os.path.getsize(p) == 0:
        continue
    info = probe(p)
    if info['video'] is None or info['duration'] < 1.5:
        continue
    key = (os.path.getsize(p), round(info['duration'], 2))  # copias subidas dos veces
    if key in seen:
        print("  duplicado, omitido:", name); continue
    seen.add(key)
    clips.append({'src': p, 'name': name, **info})
print(f"{len(clips)} clips únicos, {sum(c['duration'] for c in clips)/60:.1f} min de material")

HDR_TRC = {'arib-std-b67', 'smpte2084'}
def make_proxy(c, i):
    out = f"{WORK}/proxy/clip{i:02d}.mp4"
    c['proxy'] = out
    if os.path.exists(out) and os.path.getsize(out) > 0:
        return
    v = c['video']
    short = min(1440, min(v['width'], v['height']))         # más resolución = recorte 9:16 más nítido
    short -= short % 2
    scale = f"scale='if(gt(iw,ih),-2,{short})':'if(gt(iw,ih),{short},-2)':flags=lanczos"
    chains = []
    if v.get('color_transfer') in HDR_TRC:   # DJI HLG/HDR -> SDR Rec.709
        chains.append("zscale=t=linear:npl=100,format=gbrpf32le,zscale=p=bt709,"
                      "tonemap=hable:desat=0,zscale=t=bt709:m=bt709:r=tv,format=yuv420p," + scale)
    chains.append(scale + ",format=yuv420p")
    for vf in chains:
        try:
            run(['ffmpeg', '-y', '-v', 'error', '-i', c['src'], '-vf', f"{vf},fps={FPS}",
                 *VENC, '-g', str(FPS), '-c:a', 'aac', '-b:a', '192k', '-ac', '2', '-ar', '48000', out])
            return
        except RuntimeError as e:
            print("  reintentando sin tonemap:", str(e)[-200:])
    raise RuntimeError("No se pudo crear proxy de " + c['name'])

for i, c in enumerate(clips):
    print(f"proxy {i+1}/{len(clips)}: {c['name']}  ({c['duration']:.0f}s, {c['video']['width']}x{c['video']['height']}, trc={c['video'].get('color_transfer')})")
    make_proxy(c, i)
    cap = cv2.VideoCapture(c['proxy'])
    c['W'], c['H'] = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    c['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    c['has_audio'] = c['audio'] is not None

# ---------- B-roll: videos a proxy, imágenes tal cual ----------
BROLL = []
for k, f in enumerate(BROLL_FILES):
    b = {'src': f, 'name': os.path.basename(f), 'kind': 'image' if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')) else 'video'}
    if b['kind'] == 'video':
        info = probe(f)
        if info['video'] is None: continue
        b.update(info); bc = {'src': f, 'name': b['name'], **info}
        make_proxy(bc, 100 + k); b['proxy'] = bc['proxy']
        cap = cv2.VideoCapture(b['proxy']); b['nframes'] = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
    else:
        img = cv2.imread(f)
        if img is None: continue
        b['img'] = img; b['duration'] = 999
    BROLL.append(b)
print(f"{len(BROLL)} b-roll listos:", ', '.join(b['name'] for b in BROLL) or '—')

In [ ]:
#@title 4 · IA: transcribir lo que dices en cada clip (Whisper, palabra por palabra)
def preload_cuda12_libs():
    """faster-whisper (CTranslate2) necesita cuBLAS 12 y cuDNN 9; Colab puede traer otra versión de CUDA."""
    import ctypes, sys, site
    pat = ('libcublasLt.so.12', 'libcublas.so.12', 'libcudnn*.so.9')
    roots = site.getsitepackages() + [site.getusersitepackages()]
    find = lambda: [f for r in roots for q in pat for f in glob.glob(os.path.join(r, 'nvidia', '**', q), recursive=True)]
    if not any('libcublas.so.12' in f for f in find()):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nvidia-cublas-cu12', 'nvidia-cudnn-cu12==9.*'], check=True)
    libs = sorted(set(find()), key=lambda f: ('cublasLt' not in f, 'cublas' not in f, f))
    for _ in range(3):
        pending = []
        for f in libs:
            try: ctypes.CDLL(f, mode=ctypes.RTLD_GLOBAL)
            except OSError: pending.append(f)
        libs = pending
        if not libs: break

def split_phrases(words, max_len=7.0):
    """Corta en frases: puntuación, pausas > 0.45 s o > max_len segundos."""
    out, cur = [], []
    for w in words:
        if cur and (w[0] - cur[-1][1] > 0.45 or w[1] - cur[0][0] > max_len):
            out.append(cur); cur = []
        cur.append(w)
        if re.search(r'[.!?¿¡]$', w[2]) and w[1] - cur[0][0] > 1.2:
            out.append(cur); cur = []
    if cur: out.append(cur)
    return out

from faster_whisper import WhisperModel
try:
    preload_cuda12_libs()
except Exception as e:
    print("No pude preparar CUDA 12 para Whisper:", e)
def load_whisper(gpu=True):
    if gpu:
        try: return WhisperModel('large-v3', device='cuda', compute_type='float16')
        except Exception as e: print("Whisper en GPU no disponible:", str(e)[:200])
    print("Usando Whisper en CPU (más lento)")
    return WhisperModel('medium', device='cpu', compute_type='int8', cpu_threads=os.cpu_count())
wm = load_whisper(True)

def load_audio(path):
    """Lee el audio con ffmpeg (16 kHz mono) y se lo pasa a Whisper ya decodificado (evita el error de PyAV)."""
    raw = subprocess.run(['ffmpeg', '-v', 'error', '-i', path, '-f', 'f32le', '-ac', '1', '-ar', '16000', '-'],
                         capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.float32).copy()

def transcribe(path):
    segs, _ = wm.transcribe(load_audio(path), language=IDIOMA, word_timestamps=True, vad_filter=True,
                            initial_prompt=f"{NOMBRE_SERVER}, servidor, server.")
    return [(w.start, w.end, w.word.strip()) for sg in segs for w in (sg.words or []) if w.word.strip()]

PHRASES = []            # todas las frases de todos los clips, en orden de grabación
for ci, c in enumerate(clips):
    c['phrases'] = []
    if not c['has_audio']:
        continue
    wav = f"{WORK}/audio/clip{ci:02d}.wav"
    run(['ffmpeg', '-y', '-v', 'error', '-i', c['proxy'], '-vn', '-ac', '1', '-ar', '16000', wav])
    try:
        words = transcribe(wav)
    except RuntimeError as e:
        print("Falló Whisper en GPU, reintento en CPU:", str(e)[:200])
        wm = load_whisper(False); words = transcribe(wav)
    for k, ph in enumerate(split_phrases(words)):
        p = {'id': f"c{ci}p{k}", 'clip': ci, 'start': ph[0][0], 'end': ph[-1][1],
             'text': ' '.join(w[2] for w in ph), 'words': ph}
        c['phrases'].append(p); PHRASES.append(p)
    print(f"  {c['name']}: {len(c['phrases'])} frases · «{' '.join(w[2] for w in words)[:110]}…»")

del wm; gc.collect()                       # libera la GPU para el modelo de IA
talk_s = sum(p['end'] - p['start'] for p in PHRASES)
print(f"\n{len(PHRASES)} frases, {talk_s:.0f} s de voz en total")
if not PHRASES:
    raise RuntimeError("No se detectó voz en los clips. Revisa IDIOMA o el audio de los videos.")
open(f"{WORK}/transcripcion.txt", 'w').write("\n".join(f"{p['id']} [{clips[p['clip']]['name']} {p['start']:.1f}s] {p['text']}" for p in PHRASES))

In [ ]:
#@title 5 · IA editora: guion, palabra en grande, pantallas negras, b-roll, títulos 3D y momentos Spider-Verse
thumbs = []
for ci, c in enumerate(clips):
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, c['nframes'] // 2)
    ok, fr = cap.read(); cap.release()
    if not ok: continue
    k = 640 / max(fr.shape[:2]); fr = cv2.resize(fr, None, fx=k, fy=k)
    ok, buf = cv2.imencode('.jpg', fr, [cv2.IMWRITE_JPEG_QUALITY, 80])
    thumbs.append((ci, base64.standard_b64encode(buf.tobytes()).decode()))

listing = "\n".join(f"{p['id']} | clip {p['clip']} | {p['end'] - p['start']:.1f}s | {p['text']}" for p in PHRASES)
song_names = [os.path.basename(f) for f in SONGS]
broll_names = [b['name'] for b in BROLL]
PROMPT = f"""Eres un editor de TikTok top (estilo davecast: subtítulos amarillos con la palabra clave en grande, pantallas
negras con frases del editor, mucho b-roll, cortes cada 2-3 s) con toques de Spider-Verse. Arriba hay un fotograma de cada clip.
Abajo está la transcripción de lo que dice el creador, en frases con id. Es la promo de «{NOMBRE_SERVER}»: {DESCRIPCION_SERVER}

Arma el guion de un TikTok de unos {DURACION_OBJETIVO_S} s (entre {int(DURACION_OBJETIVO_S * 0.8)} y {int(DURACION_OBJETIVO_S * 1.1)} s sumando
duraciones; si hay menos material bueno, usa lo que valga la pena) que explique con claridad qué es, qué se hace ahí (gobierno,
roles, elecciones…), por qué entrar y cómo entrar. LA GENTE TIENE QUE QUEDARSE HASTA EL FINAL: gancho fuerte en los primeros 3 s,
una intriga que se resuelve al final, capítulos claros, sin repeticiones ni tomas falsas (si una idea se dijo varias veces o en
varios lugares, elige la mejor toma). Usa SOLO frases de la lista con su id exacto; puedes reordenarlas.

Campos por frase:
- 'enfasis': la palabra o las 2 palabras más importantes de esa frase (exactas como están en la frase); van en GRANDE en el subtítulo.
- 'tarjeta': frase del editor para una pantalla negra con letras amarillas que aparece al empezar esta frase (máx. 6 palabras,
  inteligente, intriga o remate; ej. "LUEGO TE EXPLICO POR QUÉ", "AQUÍ VIENE LO BUENO", "SÍ, TIENEN PRESIDENTE"). Una cada 15-25 s; si no, "".
- 'broll': si mientras dice esa frase conviene mostrar algo, el archivo de esta lista que corresponda: {broll_names or '(no hay b-roll)'}; si no, "".
- 'estilo_broll': "pantalla" (pantalla completa), "polaroid" (foto con marco blanco) o "tv" (dentro de una TV vieja). "" si no hay broll.
- 'capitulo': título de 1-3 palabras al empezar un tema nuevo (ej. "EL GOBIERNO", "CÓMO ENTRAR"); va en letras 3D doradas. Si no, "".
- 'efecto': "spiderverse" en 2-4 momentos de máximo impacto (revelaciones, chistes, lo más épico), "neon" en 2-4 momentos para
  subrayar algo; si no, "".
- 'pedido_cancion', 'cancion', 'segundos_cancion': si en esa frase pide al editor una canción ("editor ponga la de Colombia"):
  qué pidió tal cual, el archivo de {song_names or '(no hay canciones)'} o "", y segundos (4-15). Si no: "", "", 0.
  Las frases con pedido de canción SIEMPRE van en el guion.
También: 'gancho' (título 3D dorado del inicio, máx. 3 palabras), 'cta' (máx. 6 palabras; usa el link/Discord si lo dice),
'resumen' (1-2 frases).

Transcripción:
{listing}"""

STR = {'type': 'string'}
SCHEMA = {'type': 'object', 'additionalProperties': False, 'required': ['gancho', 'cta', 'resumen', 'guion'],
          'properties': {'gancho': STR, 'cta': STR, 'resumen': STR,
                         'guion': {'type': 'array', 'items': {'type': 'object', 'additionalProperties': False,
                                   'required': ['id', 'enfasis', 'tarjeta', 'broll', 'estilo_broll', 'capitulo', 'efecto',
                                                'pedido_cancion', 'cancion', 'segundos_cancion'],
                                   'properties': {'id': STR, 'enfasis': STR, 'tarjeta': STR, 'broll': STR, 'estilo_broll': STR,
                                                  'capitulo': STR, 'efecto': STR, 'pedido_cancion': STR, 'cancion': STR,
                                                  'segundos_cancion': {'type': 'number'}}}}}}

def ask_local_qwen(prompt, images):
    import urllib.request
    from openai import OpenAI
    vllm_bin = '/content/vllm_env/bin/vllm'
    if not os.path.exists(vllm_bin):
        raise RuntimeError("vLLM no está instalado (corre la celda 1 en Colab).")
    log = open(f"{WORK}/vllm.log", 'w')
    proc = subprocess.Popen([vllm_bin, 'serve', MODELO_LOCAL, '--port', '8000', '--max-model-len', '65536',
                             '--gpu-memory-utilization', '0.88', '--limit-mm-per-prompt', json.dumps({'image': 32}),
                             '--served-model-name', 'local'], stdout=log, stderr=subprocess.STDOUT)
    print(f"Cargando {MODELO_LOCAL} en la GPU (la 1.ª vez descarga ~56 GB, 5-15 min)…")
    t0 = time.time()
    while True:
        try:
            urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); break
        except Exception:
            if proc.poll() is not None:
                raise RuntimeError("vLLM se detuvo:\n" + open(f"{WORK}/vllm.log").read()[-3000:])
            if time.time() - t0 > 40 * 60:
                proc.terminate(); raise RuntimeError("vLLM tardó demasiado en arrancar")
            time.sleep(10)
    print(f"Modelo listo en {(time.time() - t0) / 60:.1f} min")
    try:
        client = OpenAI(base_url='http://localhost:8000/v1', api_key='local')
        content = []
        for ci, b in images:
            content += [{'type': 'text', 'text': f"Clip {ci}"},
                        {'type': 'image_url', 'image_url': {'url': f"data:image/jpeg;base64,{b}"}}]
        content.append({'type': 'text', 'text': prompt})
        r = client.chat.completions.create(
            model='local', messages=[{'role': 'user', 'content': content}], max_tokens=16000,
            temperature=0.7, top_p=0.8, presence_penalty=1.5,
            response_format={'type': 'json_schema', 'json_schema': {'name': 'guion', 'schema': SCHEMA}},
            extra_body={'top_k': 20, 'chat_template_kwargs': {'enable_thinking': False}})
        txt = r.choices[0].message.content.strip()
        return json.loads(txt[txt.find('{'): txt.rfind('}') + 1])
    finally:
        proc.terminate()
        try: proc.wait(timeout=60)
        except Exception: proc.kill()

def ask_claude(prompt, images):
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        key = os.environ.get('ANTHROPIC_API_KEY')
    if not key:
        raise RuntimeError("Falta el secreto ANTHROPIC_API_KEY")
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    content = []
    for ci, b in images:
        content += [{'type': 'text', 'text': f"Clip {ci}"},
                    {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': b}}]
    content.append({'type': 'text', 'text': prompt})
    with client.beta.messages.stream(
        model='claude-opus-5-5', max_tokens=16000,
        betas=['server-side-fallback-2026-07-01'], fallbacks='default',
        output_config={'effort': 'high', 'format': {'type': 'json_schema', 'schema': SCHEMA}},
        messages=[{'role': 'user', 'content': content}],
    ) as stream:
        msg = stream.get_final_message()
    if msg.stop_reason == 'refusal':
        raise RuntimeError("Claude declinó la petición")
    return json.loads(next(b.text for b in msg.content if b.type == 'text'))

STOP = set("""que de la el en y a los las un una por con para es no se lo me mi te tu su al del como pero más
muy ya este esta eso esto hay son está están ser hacer tiene tienen todo todos ahí aquí entonces bueno pues o si
the and to of is it in you for on this that with are""".split())
import difflib, unicodedata
def norm(s):
    return ''.join(ch for ch in unicodedata.normalize('NFD', s.lower()) if unicodedata.category(ch) != 'Mn')
SONG_REQ = re.compile(r'\b(pon|ponga|ponle|pongan|ponme|pónganle|ponganle|metele|métele|que suene|dale play a)\b(.{0,40})', re.I)
def song_request(text):
    """Texto de la canción pedida ("la de colombia…") o '' si la frase no pide canción."""
    m = SONG_REQ.search(norm(text))
    return m.group(2).strip(' ,.') if m else ''

def match_song(text):
    """Si la frase pide una canción, devuelve el archivo de /musica que mejor coincide."""
    m = SONG_REQ.search(norm(text))
    if not m or not SONGS: return ''
    req = norm(m.group(2)); best, score = '', 0.0
    for f in SONGS:
        name = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(os.path.basename(f))[0]))
        sc = max([difflib.SequenceMatcher(None, name, req).ratio()] + [1.0 for w in name.split() if len(w) > 3 and w in req])
        if sc > score: best, score = os.path.basename(f), sc
    return best if score >= 0.55 else ''

def simple_plan():
    """Sin IA: orden de grabación, hasta la duración objetivo, con algo de b-roll y efectos repartidos."""
    plan, tot = [], 0.0
    for k, p in enumerate(PHRASES):
        req = song_request(p['text'])
        if len(p['words']) < 3 and not req: continue
        words = [re.sub(r'[^\wáéíóúñü]', '', w[2].lower()) for w in p['words']]
        kw = max((w for w in words if w not in STOP), key=len, default='')
        n = len(plan)
        plan.append({'id': p['id'], 'enfasis': kw, 'tarjeta': '', 'capitulo': '',
                     'broll': BROLL[n // 4 % len(BROLL)]['name'] if BROLL and n % 4 == 3 else '',
                     'estilo_broll': ['polaroid', 'tv', 'pantalla'][n // 4 % 3] if BROLL and n % 4 == 3 else '',
                     'efecto': 'spiderverse' if n % 9 == 5 else ('neon' if n % 9 == 2 else ''),
                     'pedido_cancion': req, 'cancion': match_song(p['text']), 'segundos_cancion': SEGUNDOS_CANCION_PEDIDA if req else 0})
        tot += p['end'] - p['start']
        if tot >= DURACION_OBJETIVO_S: break
    return {'gancho': NOMBRE_SERVER.upper(), 'cta': 'ÚNETE A LA REPÚBLICA', 'resumen': 'Guion automático (sin IA).', 'guion': plan}

PLAN = None
try:
    if MOTOR_IA == 'local_qwen': PLAN = ask_local_qwen(PROMPT, thumbs)
    elif MOTOR_IA == 'claude': PLAN = ask_claude(PROMPT, thumbs)
except Exception as e:
    print(f"⚠ {MOTOR_IA} falló, uso el guion automático:", str(e)[-1500:])

byid = {p['id']: p for p in PHRASES}
if PLAN:
    seen = set()
    PLAN['guion'] = [g for g in PLAN['guion'] if g['id'] in byid and not (g['id'] in seen or seen.add(g['id']))]
if not PLAN or not PLAN['guion']:
    PLAN = simple_plan()
song_files = {os.path.basename(f): f for f in SONGS}
broll_by = {b['name']: b for b in BROLL}
DEF = (('enfasis', ''), ('tarjeta', ''), ('broll', ''), ('estilo_broll', ''), ('capitulo', ''), ('efecto', ''),
       ('pedido_cancion', ''), ('cancion', ''), ('segundos_cancion', 0))
for g in PLAN['guion']:
    for k_, v_ in DEF: g.setdefault(k_, v_)
    txt_ = byid[g['id']]['text']
    if not g['pedido_cancion'].strip(): g['pedido_cancion'] = song_request(txt_)
    if g['cancion'] not in song_files: g['cancion'] = match_song(txt_)
    if MODO_CANCIONES != 'usar_archivos': g['cancion'] = ''
    g['segundos_cancion'] = float(SEGUNDOS_CANCION_PEDIDA) if g['pedido_cancion'] else 0.0
    if g['broll'] not in broll_by: g['broll'] = ''
    if g['broll'] and g['estilo_broll'] not in ('pantalla', 'polaroid', 'tv'): g['estilo_broll'] = 'pantalla'
    if g['estilo_broll'] == 'tv' and not TV_VIEJA: g['estilo_broll'] = 'polaroid'
    if g['efecto'] == 'spiderverse' and not SPIDERVERSE: g['efecto'] = ''
    if g['efecto'] == 'neon' and not NEON: g['efecto'] = ''
have = {g['id'] for g in PLAN['guion']}
for p in PHRASES:
    req = song_request(p['text'])
    if req and p['id'] not in have:
        print(f"  + añado el pedido de canción «{p['text']}»")
        pos = next((i for i, g in enumerate(PLAN['guion']) if PHRASES.index(byid[g['id']]) > PHRASES.index(p)), len(PLAN['guion']))
        g = dict(DEF); g.update({'id': p['id'], 'pedido_cancion': req, 'segundos_cancion': float(SEGUNDOS_CANCION_PEDIDA),
                                 'cancion': match_song(p['text']) if MODO_CANCIONES == 'usar_archivos' else ''})
        PLAN['guion'].insert(pos, g)
def _len(): return sum(byid[g['id']]['end'] - byid[g['id']]['start'] + g['segundos_cancion'] for g in PLAN['guion'])
while len(PLAN['guion']) > 3 and _len() > DURACION_OBJETIVO_S * 1.25:
    drop = next((i for i in range(len(PLAN['guion']) - 2, 0, -1) if not PLAN['guion'][i]['pedido_cancion']), None)
    if drop is None: break
    PLAN['guion'].pop(drop)
if TEXTO_CTA.strip(): PLAN['cta'] = TEXTO_CTA.strip()

print(f"Gancho: {PLAN['gancho']}  ·  CTA: {PLAN['cta']}  ·  {len(PLAN['guion'])} frases, ~{_len():.0f}s\n")
for g in PLAN['guion']:
    if g['capitulo']: print(f"\n  ── ✨ {g['capitulo'].upper()} ──")
    if g['tarjeta']: print(f"  ⬛ {g['tarjeta'].upper()}")
    extra = ' '.join(x for x in (f"🎞 {g['broll']} ({g['estilo_broll']})" if g['broll'] else '',
                                 {'spiderverse': '🕷 SPIDER-VERSE', 'neon': '💡 NEÓN'}.get(g['efecto'], ''),
                                 f"🎵 {g['pedido_cancion']} → {g['cancion'] or 'TikTok'}" if g['pedido_cancion'] else '') if x)
    print(f"  {byid[g['id']]['text']}   [{g['enfasis']}]  {extra}")

In [ ]:
#@title 6 · Línea de tiempo: cortes, pantallas negras, b-roll, títulos 3D, efectos y audio (con entradas suaves y whooshes)
HOOK_S, CTA_S, CARD_S = 2.6, 3.0, 1.7
PIECES = []
for g in PLAN['guion']:
    p = byid[g['id']]; c = clips[p['clip']]
    s0 = max(0.0, p['start'] - 0.08); s1 = min(c['duration'], p['end'] + 0.15)
    song = song_files.get(g['cancion']) if g['cancion'] else None
    req = g['pedido_cancion'].strip(); hold = float(g['segundos_cancion']) if req else 0.0
    prev = PIECES[-1] if PIECES else None
    if prev and not prev['req'] and not req and prev['clip'] == p['clip'] and 0 <= s0 - prev['s1'] < 0.3:
        prev['s1'] = s1; prev['phr'].append((p, g))
    else:
        PIECES.append({'clip': p['clip'], 's0': s0, 's1': s1, 'phr': [(p, g)], 'song': song, 'req': req, 'hold': hold})

t = 0
WORDS, CAPS, CARDS, BROLLS, TITLES, FX, SONG_EVENTS, SONG_MARKS, SFX = [], [], [], [], [], [], [], [], []
for k, pc in enumerate(PIECES):
    pc['idx'] = k
    pc['src_f0'] = int(round(pc['s0'] * FPS))
    pc['n_talk'] = max(1, int(round((pc['s1'] - pc['s0']) * FPS)))
    pc['n'] = pc['n_talk'] + int(round(pc['hold'] * FPS))
    pc['f0'] = t; pc['f1'] = t + pc['n']; t = pc['f1']
    if pc['req']:
        ev = {'file': pc['song'], 'name': pc['req'], 't0': (pc['f0'] + pc['n_talk']) / FPS, 'dur': pc['hold']}
        (SONG_EVENTS if pc['song'] else SONG_MARKS).append(ev)
    for p, g in pc['phr']:
        off = pc['f0'] / FPS - pc['s0']
        ws = [(a + off, b + off, w) for a, b, w in p['words']]; WORDS += ws
        emph = {norm(re.sub(r'[^\w]', '', x)) for x in g['enfasis'].split()}
        # subtítulos: grupos de 2-5 palabras; la de énfasis va grande
        grp = []
        for w in ws:
            if grp and (len(grp) >= 5 or w[0] - grp[-1][1] > 0.45 or (len(grp) >= 2 and grp[-1][2].endswith((',', '.', '?', '!')))):
                CAPS.append(grp); grp = []
            grp.append((w[0], w[1], w[2], norm(re.sub(r'[^\w]', '', w[2])) in emph))
        if grp: CAPS.append(grp)
        fs, fe = int((p['start'] + off) * FPS), int((p['end'] + off) * FPS)
        if g['tarjeta'].strip():
            CARDS.append({'f0': fs, 'f1': fs + int(CARD_S * FPS), 'text': g['tarjeta'].strip().upper()})
        if g['broll']:
            b0 = fs + (int(CARD_S * FPS) if g['tarjeta'].strip() else int(0.25 * FPS))
            b1 = max(b0 + int(2.2 * FPS), min(fe, b0 + int(5 * FPS)))
            BROLLS.append({'f0': b0, 'f1': min(b1, pc['f1']), 'b': broll_by[g['broll']], 'style': g['estilo_broll']})
        if g['capitulo'].strip() and TITULOS_3D_DORADOS:
            TITLES.append({'f0': fs, 'f1': fs + int(2.4 * FPS), 'text': g['capitulo'].strip().upper(), 'piece': k, 'big': True})
        if g['efecto'] in ('spiderverse', 'neon'):
            dur = min(3.0, max(1.4, (fe - fs) / FPS)) if g['efecto'] == 'spiderverse' else 1.6
            FX.append({'kind': g['efecto'], 'f0': fs, 'f1': fs + int(dur * FPS)})
TOTAL_FRAMES = t; L = TOTAL_FRAMES / FPS
if TITULOS_3D_DORADOS:
    TITLES.insert(0, {'f0': 0, 'f1': int(min(HOOK_S, PIECES[0]['n'] / FPS) * FPS), 'text': PLAN['gancho'].upper(), 'piece': 0, 'big': True})
CTA = {'f0': TOTAL_FRAMES - int(CTA_S * FPS), 'f1': TOTAL_FRAMES, 'text': PLAN['cta'].upper()}
for w in TITLES:
    pc = PIECES[w['piece']]; w['f0'] = max(w['f0'], pc['f0']); w['f1'] = min(w['f1'], pc['f1'])
TITLES = [w for w in TITLES if w['f1'] - w['f0'] >= 15]
# nada encima de nada: tarjetas mandan; luego b-roll; títulos 3D solo sobre la persona hablando
def clash(a, lst, pad=0): return any(a['f0'] < b['f1'] + pad and b['f0'] < a['f1'] + pad for b in lst)
CARDS = [c_ for i, c_ in enumerate(CARDS) if not clash(c_, CARDS[:i], FPS * 4) and c_['f1'] < CTA['f0']]
BROLLS = [b for i, b in enumerate(BROLLS) if not clash(b, CARDS) and not clash(b, BROLLS[:i]) and b['f1'] <= CTA['f0']]
TITLES = [w for w in TITLES if not clash(w, CARDS) and not clash(w, BROLLS)]
FX = [x for i, x in enumerate(FX) if not clash(x, CARDS) and not clash(x, FX[:i], FPS * 3)]

# ritmo: cambio de encuadre cada ~2.5 s (alineado a palabra) como en davecast
for pc in PIECES:
    starts = sorted(int(round(a * FPS)) - pc['f0'] for a, _, _ in WORDS if pc['f0'] <= a * FPS < pc['f0'] + pc['n_talk'])
    marks, last = [0], 0
    for s_ in starts:
        if s_ - last >= int(2.4 * FPS) and pc['n'] - s_ >= int(1.2 * FPS): marks.append(s_); last = s_
    pc['zmarks'] = marks
    lv = [1.0, 1.16, 1.06, 1.24] if pc['idx'] % 2 == 0 else [1.12, 1.0, 1.2, 1.05]
    pc['zlevels'] = [lv[j % 4] for j in range(len(marks))]
    pc['xoff'] = [(-0.04, 0.0, 0.04, 0.0)[j % 4] for j in range(len(marks))]   # leve cambio de lado = "otro ángulo"

for x in CARDS: SFX.append(('whoosh', x['f0'] / FPS))
for x in BROLLS: SFX.append(('whoosh', x['f0'] / FPS))
for x in TITLES: SFX.append(('hit', x['f0'] / FPS))
for x in FX: SFX.append(('glitch' if x['kind'] == 'spiderverse' else 'shine', x['f0'] / FPS))
for e in SONG_EVENTS + SONG_MARKS: SFX.append(('riser', e['t0'] - 0.9))
print(f"{len(PIECES)} planos · {L / 60:.1f} min · {len(CAPS)} subtítulos · {len(CARDS)} pantallas negras · {len(BROLLS)} b-roll · "
      f"{len(TITLES)} títulos 3D · {sum(x['kind'] == 'spiderverse' for x in FX)} Spider-Verse · {sum(x['kind'] == 'neon' for x in FX)} neón · "
      f"{len(SONG_EVENTS)} canciones dentro · {len(SONG_MARKS)} para TikTok")

# ---------- Audio ----------
SR = 48000
parts = []
for pc in PIECES:
    c = clips[pc['clip']]; d = pc['n'] / FPS; dt = pc['n_talk'] / FPS; p = f"{WORK}/audio/piece{pc['idx']:03d}.wav"
    src = ['-f', 'lavfi', '-i', 'anullsrc=r=48000:cl=stereo'] if not c['has_audio'] else ['-ss', f"{pc['src_f0'] / FPS:.3f}", '-i', c['proxy']]
    duck = f",volume='1-0.75*clip((t-{dt:.3f})/0.4,0,1)':eval=frame" if pc['req'] else ""
    run(['ffmpeg', '-y', '-v', 'error', *src, '-vn', '-ac', '2', '-ar', str(SR),
         '-af', f"apad,atrim=0:{d:.4f}{duck},afade=t=in:d=0.02,afade=t=out:st={max(0, d - 0.03):.4f}:d=0.03", p])
    parts.append(p)
open(f"{WORK}/alist.txt", 'w').writelines(f"file '{p}'\n" for p in parts)
VOICE_WAV = f"{WORK}/voz.wav"
run(['ffmpeg', '-y', '-v', 'error', '-f', 'concat', '-safe', '0', '-i', f"{WORK}/alist.txt", '-c', 'copy', VOICE_WAV])
VOICE_PROC = f"{WORK}/voz_limpia.wav"
run(['ffmpeg', '-y', '-v', 'error', '-i', VOICE_WAV, '-af',
     "highpass=f=80,afftdn=nf=-25,acompressor=threshold=-20dB:ratio=3:attack=5:release=80,"
     f"loudnorm=I=-14:TP=-1.5:LRA=7,aresample={SR},apad,atrim=0:{L:.4f}", '-ar', str(SR), VOICE_PROC])

# Efectos de sonido sintetizados (whoosh, golpe, glitch, brillo, subida antes de la canción)
def synth(kind):
    rng = np.random.default_rng({'whoosh': 1, 'hit': 2, 'glitch': 3, 'shine': 4, 'riser': 5}[kind])
    if kind == 'riser':
        n = int(0.9 * SR); tt = np.arange(n) / SR; nz = rng.standard_normal(n)
        tone = np.sin(2 * np.pi * (200 * tt + 900 * tt ** 2)) * 0.25
        return ((nz * 0.15 + tone) * (tt / 0.9) ** 2).astype(np.float32)
    if kind == 'hit':
        n = int(0.6 * SR); tt = np.arange(n) / SR
        return (np.sin(2 * np.pi * 55 * tt) * np.exp(-tt * 7) * 0.9 + rng.standard_normal(n) * np.exp(-tt * 30) * 0.3).astype(np.float32)
    if kind == 'glitch':
        n = int(0.35 * SR); x = rng.standard_normal(n) * 0.3
        x *= np.repeat(rng.random(n // 600 + 1) > 0.4, 600)[:n]
        return (np.sign(x) * np.abs(x) ** 0.5 * 0.35).astype(np.float32)
    if kind == 'shine':
        n = int(0.7 * SR); tt = np.arange(n) / SR
        return (sum(np.sin(2 * np.pi * f * tt) for f in (1760, 2637, 3520)) / 3 * np.exp(-tt * 5) * 0.25).astype(np.float32)
    n = int(0.55 * SR); tt = np.arange(n) / SR; nz = rng.standard_normal(n)
    env = np.sin(np.pi * tt / tt[-1]) ** 2
    out = np.zeros(n, np.float32); y = 0.0
    for i in range(n):                                   # filtro paso bajo que barre (el "fiuuu")
        a_ = 0.02 + 0.5 * env[i]; y += a_ * (nz[i] - y); out[i] = y
    return out * env * 0.6
SFX_WAV = f"{WORK}/sfx.wav"
sfx_track = np.zeros((int(L * SR) + SR, ), np.float32)
if EFECTOS_SONIDO:
    bank = {}
    for kind, t0 in SFX:
        if t0 < 0: continue
        snd = bank.setdefault(kind, synth(kind)); i0 = int(t0 * SR)
        sfx_track[i0:i0 + len(snd)] += snd[:max(0, len(sfx_track) - i0)]
import wave
with wave.open(SFX_WAV, 'wb') as wf:
    wf.setnchannels(1); wf.setsampwidth(2); wf.setframerate(SR)
    wf.writeframes((np.clip(sfx_track[:int(L * SR)], -1, 1) * 32767 * 0.5).astype(np.int16).tobytes())

# Música de fondo: lista en orden con fundidos largos
def find_song(name):
    want = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(name.strip())[0])); best, score = None, 0.0
    for f in SONGS:
        have_ = norm(re.sub(r'[_\-]+', ' ', os.path.splitext(os.path.basename(f))[0]))
        sc = 1.0 if want == have_ else difflib.SequenceMatcher(None, want, have_).ratio()
        if sc > score: best, score = f, sc
    return best if score >= 0.6 else None
BG_FILES = []
for name in [n for n in MUSICA_FONDO.split(',') if n.strip()]:
    f = find_song(name)
    print(f"  fondo: «{name.strip()}» → {os.path.basename(f) if f else 'NO ENCONTRADA'}")
    if f: BG_FILES.append(f)
BG_WAV = f"{WORK}/fondo.wav"
if BG_FILES:
    ins = sum((['-i', f] for f in BG_FILES), [])
    chain = [';'.join(f"[{i}:a]aresample={SR},aformat=channel_layouts=stereo[b{i}]" for i in range(len(BG_FILES)))]; last = '[b0]'
    for i in range(1, len(BG_FILES)):
        chain.append(f"{last}[b{i}]acrossfade=d=4:c1=qsin:c2=qsin[x{i}]"); last = f'[x{i}]'
    run(['ffmpeg', '-y', '-v', 'error', *ins, '-filter_complex', ';'.join(chain), '-map', last, '-ar', str(SR), BG_WAV])
else:
    run(['ffmpeg', '-y', '-v', 'error', '-f', 'lavfi', '-i', f'anullsrc=r={SR}:cl=stereo', '-t', '1', BG_WAV])

# El fondo baja suave 0.6 s antes de cada canción pedida y vuelve en 1 s después (sin cortes bruscos)
ramps = [f"clip((t-{e['t0'] - 0.6:.2f})/0.6,0,1)*clip(({e['t0'] + e['dur'] + 1.0:.2f}-t)/1.0,0,1)" for e in SONG_EVENTS + SONG_MARKS]
env = f"max(0,1-({'+'.join(ramps)}))" if ramps else "1"
inputs = ['-i', VOICE_PROC, '-stream_loop', '-1', '-i', BG_WAV, '-i', SFX_WAV]
fc = ["[0:a]asplit=2[v1][v2]",
      f"[1:a]atrim=0:{L:.3f},asetpts=N/SR/TB,volume={VOLUMEN_MUSICA},volume='{env}':eval=frame,"
      f"afade=t=in:d=1.0,afade=t=out:st={max(0, L - 2):.3f}:d=2[m]",
      "[m][v2]sidechaincompress=threshold=0.03:ratio=6:attack=30:release=500[md]",
      "[2:a]aformat=channel_layouts=stereo[fx]"]
mix_in = ['[v1]', '[md]', '[fx]']
for j, e in enumerate(SONG_EVENTS):
    inputs += ['-i', e['file']]
    sdur = float(probe(e['file'])['duration'] or 60)
    ss = min(max(0.0, sdur * 0.3), max(0.0, sdur - e['dur']))
    fc.append(f"[{j + 3}:a]aresample={SR},aformat=channel_layouts=stereo,atrim={ss:.2f}:{ss + e['dur'] + 1:.2f},asetpts=N/SR/TB,"
              f"volume={VOLUMEN_CANCIONES},afade=t=in:d=0.5:curve=qsin,afade=t=out:st={max(0, e['dur'] - 0.2):.2f}:d=1.2:curve=qsin,"
              f"adelay={int((e['t0'] - 0.25) * 1000)}|{int((e['t0'] - 0.25) * 1000)}[s{j}]")
    mix_in.append(f'[s{j}]')
fc.append(f"{''.join(mix_in)}amix=inputs={len(mix_in)}:normalize=0:duration=longest,alimiter=limit=0.95,apad,atrim=0:{L:.4f}[o]")
MIX_WAV = f"{WORK}/mix.wav"
run(['ffmpeg', '-y', '-v', 'error', *inputs, '-filter_complex', ';'.join(fc), '-map', '[o]', '-ar', str(SR), '-t', f"{L:.3f}", MIX_WAV])
print("Audio listo:", MIX_WAV)

In [ ]:
#@title 7 · Look: color de cine, subtítulos amarillos, pantallas negras, 3D dorado, polaroid, TV vieja, neón y Spider-Verse
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import urllib.request
FONT_DIR = f"{WORK}/fonts"; os.makedirs(FONT_DIR, exist_ok=True)
GF = 'https://github.com/google/fonts/raw/main/'
FONT_URLS = {'Anton-Regular.ttf': GF + 'ofl/anton/Anton-Regular.ttf',
             'BebasNeue-Regular.ttf': GF + 'ofl/bebasneue/BebasNeue-Regular.ttf',
             'Cinzel.ttf': GF + 'ofl/cinzel/Cinzel%5Bwght%5D.ttf',
             'VT323-Regular.ttf': GF + 'ofl/vt323/VT323-Regular.ttf',
             'Bangers-Regular.ttf': GF + 'ofl/bangers/Bangers-Regular.ttf'}
for fn, url in FONT_URLS.items():
    p = os.path.join(FONT_DIR, fn)
    if not os.path.exists(p):
        try: urllib.request.urlretrieve(url, p)
        except Exception as e: print("No pude bajar", fn, e)
def font(fn, size, *alts):
    for cand in [os.path.join(FONT_DIR, f) for f in (fn,) + alts] + ['/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf']:
        try:
            f = ImageFont.truetype(cand, size)
            if 'Cinzel' in cand:
                try: f.set_variation_by_axes([800])     # Cinzel en negrita
                except Exception: pass
            return f
        except Exception: pass
    return ImageFont.load_default(size=size)

# ---------- Color de cine cálido (verdes suaves) + grano + viñeta ----------
def curve(points):
    xs, ys = zip(*points); return np.clip(np.interp(np.arange(256), xs, ys), 0, 255).astype(np.uint8)
LB, LG, LR = (curve([(0, 14), (70, 70), (150, 140), (255, 228)]), curve([(0, 8), (128, 124), (255, 244)]),
              curve([(0, 6), (90, 92), (190, 200), (255, 255)]))
def grade(fr):
    b, g, r = cv2.split(fr); fr = cv2.merge([cv2.LUT(b, LB), cv2.LUT(g, LG), cv2.LUT(r, LR)])
    hsv = cv2.cvtColor(fr, cv2.COLOR_BGR2HSV).astype(np.int16)
    green = (hsv[..., 0] > 30) & (hsv[..., 0] < 85)
    hsv[..., 1] = np.where(green, hsv[..., 1] * 0.78, hsv[..., 1] * 0.97)
    return cv2.cvtColor(np.clip(hsv, 0, 255).astype(np.uint8), cv2.COLOR_HSV2BGR)
_vig = None; _grain = [np.random.normal(0, 5, (480, 270)).astype(np.float32) for _ in range(8)]; _gi = [0]
def finish(fr):
    global _vig
    if _vig is None:
        yy, xx = np.mgrid[0:OUT_H, 0:OUT_W]; d = np.sqrt(((xx - OUT_W / 2) / (OUT_W / 2)) ** 2 + ((yy - OUT_H / 2) / (OUT_H / 2)) ** 2)
        _vig = (1 - 0.28 * np.clip(d - 0.5, 0, 1) ** 1.4)[..., None].astype(np.float32)
    _gi[0] += 1; n = cv2.resize(_grain[_gi[0] % 8], (OUT_W, OUT_H))[..., None]
    return np.clip(fr.astype(np.float32) * _vig + n, 0, 255).astype(np.uint8)

def paste_rgba(frame, rgba, x0, y0, alpha=1.0):
    h, w = rgba.shape[:2]; H, W = frame.shape[:2]
    xa, ya, xb, yb = max(0, x0), max(0, y0), min(W, x0 + w), min(H, y0 + h)
    if xa >= xb or ya >= yb: return
    sub = rgba[ya - y0: yb - y0, xa - x0: xb - x0]
    a = sub[..., 3:4].astype(np.float32) / 255 * alpha
    frame[ya:yb, xa:xb] = (frame[ya:yb, xa:xb] * (1 - a) + sub[..., [2, 1, 0]] * a).astype(np.uint8)

# ---------- Persona: encuadre y máscara ----------
from ultralytics import YOLO
seg_model = YOLO('yolo11l-seg.pt')
def person_mask(fr):
    r = seg_model.predict(fr, classes=[0], conf=0.25, verbose=False, retina_masks=True, imgsz=640)[0]
    if r.masks is None or not len(r.masks): return np.zeros(fr.shape[:2], np.float32)
    m = r.masks.data.cpu().numpy().max(0).astype(np.float32)
    if m.shape != fr.shape[:2]: m = cv2.resize(m, (fr.shape[1], fr.shape[0]))
    return cv2.GaussianBlur(cv2.dilate(m, np.ones((7, 7), np.uint8)), (0, 0), 3)
def track_centers(pc):
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0'])
    xs, ts = [], []
    for i in range(n):
        ok, fr = cap.read()
        if not ok: break
        if i % 4 == 0:
            r = seg_model.predict(fr, classes=[0], conf=0.3, verbose=False, imgsz=640)[0]
            if len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy(); a = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
                j = int(a.argmax()); xs.append(float((b[j, 0] + b[j, 2]) / 2 / fr.shape[1])); ts.append(i)
    cap.release()
    if not xs: return np.full(n, 0.5)
    raw = np.interp(np.arange(n), ts, xs); out = np.empty(n); out[0] = np.median(raw[:15])
    for i in range(1, n): out[i] = out[i - 1] * 0.94 + raw[i] * 0.06
    return out
def crop_frame(fr, cxn, zoom, W=OUT_W, H=OUT_H, ytop=0.35):
    h, w = fr.shape[:2]; tgt = W / H
    cw, ch = (h * tgt, h) if w / h > tgt else (w, w / tgt)
    cw, ch = cw / zoom, ch / zoom
    x0 = np.clip(cxn * w - cw / 2, 0, w - cw); y0 = np.clip((h - ch) * ytop, 0, h - ch)
    M = np.float32([[W / cw, 0, -x0 * W / cw], [0, H / ch, -y0 * H / ch]])
    return cv2.warpAffine(fr, M, (W, H), flags=cv2.INTER_CUBIC)
for pc in PIECES:
    c = clips[pc['clip']]
    pc['cx'] = track_centers(pc) if c['W'] / c['H'] > OUT_W / OUT_H else np.full(pc['n'], 0.5)

# ---------- Subtítulos amarillos estilo davecast (palabra clave GRANDE, con brillo) ----------
YELLOW = (255, 196, 18)
def glow_text_rgba(lines, color=YELLOW, glow=0.85):
    """lines = [(texto, tamaño)]; centrado, con brillo amarillo y sombra suave."""
    fonts = [font('Anton-Regular.ttf', sz, 'BebasNeue-Regular.ttf') for _, sz in lines]
    widths = [f.getlength(t) for (t, _), f in zip(lines, fonts)]
    W = int(max(widths)) + 120; H = sum(int(sz * 1.08) for _, sz in lines) + 120
    img = Image.new('RGBA', (W, H), (0, 0, 0, 0)); d = ImageDraw.Draw(img); y = 60
    for (t, sz), f, wd in zip(lines, fonts, widths):
        d.text(((W - wd) / 2, y), t, font=f, fill=color + (255,)); y += int(sz * 1.08)
    a = np.array(img)
    al = a[..., 3].astype(np.float32) / 255
    halo = cv2.GaussianBlur(al, (0, 0), 14) * glow
    shadow = cv2.GaussianBlur(al, (0, 0), 5) * 0.7
    out = np.zeros_like(a, dtype=np.float32)
    out[..., :3] = np.array(color, np.float32)
    comb = np.maximum(al, halo)
    rgb = out[..., :3] * (comb[..., None] > 0) * 1.0
    # sombra oscura debajo del brillo para que se lea sobre fondos claros
    base = np.dstack([np.zeros_like(al)] * 3 + [shadow])
    top = np.dstack([rgb, comb])
    a_out = top[..., 3] + base[..., 3] * (1 - top[..., 3])
    rgb_out = (top[..., :3] * top[..., 3:4]) / np.maximum(a_out[..., None], 1e-6)
    return np.dstack([rgb_out, a_out * 255]).clip(0, 255).astype(np.uint8)

_cap_cache = {}
def caption_rgba(grp):
    key = tuple(w[2] for w in grp) + tuple(w[3] for w in grp)
    if key in _cap_cache: return _cap_cache[key]
    clean = lambda s: re.sub(r'[{}\\]', '', s).upper().strip(' ,')
    big = [clean(w[2]) for w in grp if w[3]]; small = [clean(w[2]) for w in grp if not w[3]]
    if big:
        lines = ([(' '.join(small), 62)] if small else []) + [(' '.join(big), 132 if len(' '.join(big)) <= 10 else 104)]
    else:
        txt = ' '.join(clean(w[2]) for w in grp)
        lines = [(txt, 84 if len(txt) <= 16 else 66)]
    r = glow_text_rgba(lines); _cap_cache[key] = r
    return r
def draw_caption(frame, t):
    for grp in CAPS:
        if grp[0][0] <= t < grp[-1][1] + 0.12:
            r = caption_rgba(grp); h, w = r.shape[:2]
            pop = min(1.0, (t - grp[0][0]) / 0.12)
            if pop < 1:
                s_ = 0.86 + 0.14 * pop; r = cv2.resize(r, None, fx=s_, fy=s_); h, w = r.shape[:2]
            if w > OUT_W * 0.92:
                k_ = OUT_W * 0.92 / w; r = cv2.resize(r, None, fx=k_, fy=k_); h, w = r.shape[:2]
            paste_rgba(frame, r, (OUT_W - w) // 2, int(OUT_H * 0.70) - h // 2)
            return

# ---------- Pantalla negra con textura (notas del editor) ----------
def grunge_bg(seed=0):
    rng = np.random.default_rng(seed)
    base = cv2.GaussianBlur(rng.random((OUT_H // 8, OUT_W // 8)).astype(np.float32), (0, 0), 3)
    base = cv2.resize(base, (OUT_W, OUT_H)) * 22 + 8
    base += rng.normal(0, 4, (OUT_H, OUT_W))
    img = np.dstack([base] * 3).clip(0, 255).astype(np.uint8)
    for _ in range(40):                                  # rayones y polvo
        x, y = int(rng.integers(OUT_W)), int(rng.integers(OUT_H)); l = int(rng.integers(20, 160))
        ang = rng.uniform(0, np.pi)
        cv2.line(img, (x, y), (int(x + l * np.cos(ang)), int(y + l * np.sin(ang))), (60, 60, 60), 1, cv2.LINE_AA)
    return img
GRUNGE = [grunge_bg(i) for i in range(3)]
_card_cache = {}
def draw_card(frame, card, g):
    t = (g - card['f0']) / FPS; T = (card['f1'] - card['f0']) / FPS
    key = card['text']
    if key not in _card_cache:
        words = card['text'].split()
        first, rest = words[0], words[1:]
        lines = [(first, 210 if len(first) <= 8 else 150)]
        mid = (len(rest) + 1) // 2
        if rest: lines.append((' '.join(rest[:mid]), 96))
        if len(rest) > 1: lines.append((' '.join(rest[mid:]), 70))
        _card_cache[key] = glow_text_rgba(lines, glow=1.0)
    r = _card_cache[key]
    frame[:] = GRUNGE[hash(key) % 3]
    s_ = 1.0 + 0.05 * t / T
    shake = int(10 * max(0, 1 - t / 0.15)) * (1 if g % 2 else -1)
    rr = cv2.resize(r, None, fx=s_, fy=s_)
    if rr.shape[1] > OUT_W * 0.92: k_ = OUT_W * 0.92 / rr.shape[1]; rr = cv2.resize(rr, None, fx=k_, fy=k_)
    h, w = rr.shape[:2]
    flick = 0.85 + 0.15 * (np.random.rand() > 0.08)
    paste_rgba(frame, rr, (OUT_W - w) // 2 + shake, (OUT_H - h) // 2, flick * min(1, t / 0.08, (T - t) / 0.12))

# ---------- Texto 3D dorado ----------
def gold3d_rgba(text, size=150):
    f = font('Cinzel.ttf', size, 'BebasNeue-Regular.ttf')
    words = text.split(); lines, cur = [], ''
    for w in words:
        if cur and len(cur) + 1 + len(w) > 12: lines.append(cur); cur = w
        else: cur = (cur + ' ' + w).strip()
    lines.append(cur)
    depth = max(8, size // 12); pad = size // 2
    W = int(max(f.getlength(l) for l in lines)) + 2 * pad + depth; lh = int(size * 1.15); H = lh * len(lines) + 2 * pad + depth
    mask = Image.new('L', (W, H), 0); d = ImageDraw.Draw(mask)
    for k, l in enumerate(lines):
        d.text(((W - depth - f.getlength(l)) / 2, pad + k * lh), l, font=f, fill=255)
    m = np.array(mask).astype(np.float32) / 255
    out = np.zeros((H, W, 4), np.float32)
    for i in range(depth, 0, -1):                        # extrusión (cara lateral marrón-dorada)
        sh = np.roll(np.roll(m, i, 0), i, 1); c_ = 0.35 + 0.35 * (1 - i / depth)
        col = np.array([150, 95, 25]) * c_
        out[..., :3] = out[..., :3] * (1 - sh[..., None]) + col * sh[..., None]
        out[..., 3] = np.maximum(out[..., 3], sh)
    yy = np.linspace(0, 1, H)[:, None]
    grad = np.stack([255 - 30 * yy, 236 - 120 * yy, 160 - 140 * yy], -1) * np.ones((1, W, 1))   # oro: claro arriba, ámbar abajo
    bl = cv2.GaussianBlur(m, (0, 0), 2.2)
    bevel = np.clip(cv2.Sobel(bl, cv2.CV_32F, 0, 1, ksize=3) * -2.5 + cv2.Sobel(bl, cv2.CV_32F, 1, 0, ksize=3) * -1.5, -0.6, 0.9)[..., None]
    face = np.clip(grad * (1 + bevel), 0, 255)
    out[..., :3] = out[..., :3] * (1 - m[..., None]) + face * m[..., None]
    out[..., 3] = np.maximum(out[..., 3], m)
    halo = cv2.GaussianBlur(out[..., 3], (0, 0), size / 7) * 0.55   # brillo cálido
    a = np.maximum(out[..., 3], halo)
    rgb = np.where(out[..., 3:4] > 0.02, out[..., :3], np.array([255, 170, 60], np.float32))
    res = np.dstack([rgb, a * 255]).clip(0, 255).astype(np.uint8)
    ys, xs = np.nonzero(res[..., 3] > 10)                       # recorta el margen vacío
    return res[max(0, ys.min() - 6): ys.max() + 7, max(0, xs.min() - 6): xs.max() + 7] if len(xs) else res

def plane_quad(w, h, cx, cy, yaw, pitch, scale, focal=1600):
    pts = np.array([[-w / 2, -h / 2, 0], [w / 2, -h / 2, 0], [w / 2, h / 2, 0], [-w / 2, h / 2, 0]], np.float32) * scale
    cy_, sy = math.cos(math.radians(yaw)), math.sin(math.radians(yaw))
    cp, sp = math.cos(math.radians(pitch)), math.sin(math.radians(pitch))
    R = np.array([[cy_, 0, sy], [0, 1, 0], [-sy, 0, cy_]]) @ np.array([[1, 0, 0], [0, cp, -sp], [0, sp, cp]])
    P = pts @ R.T; z = P[:, 2] + focal
    return np.stack([cx + P[:, 0] * focal / z, cy + P[:, 1] * focal / z], 1).astype(np.float32)
def fit_quad(q):
    x0, x1 = q[:, 0].min(), q[:, 0].max(); lo, hi = OUT_W * 0.05, OUT_W * 0.95
    if x1 - x0 > hi - lo:
        c = q.mean(0); q = (q - c) * ((hi - lo) / (x1 - x0)) + c; x0, x1 = q[:, 0].min(), q[:, 0].max()
    q = q.copy(); q[:, 0] += max(0, lo - x0) - max(0, x1 - hi); q[:, 1] += max(0, OUT_H * 0.07 - q[:, 1].min())
    return q

class SceneText:
    """Título 3D dorado colocado en el fondo, detrás de ti, siguiendo la cámara (como 'STAY CREATIVE' en el cielo)."""
    def __init__(self, spec, frame, mask):
        self.rgba = gold3d_rgba(spec['text']); h, w = self.rgba.shape[:2]
        bg = 1 - mask; scale = min(0.86 * OUT_W / w, 0.2 * OUT_H / h); best = None
        for cy in (0.13, 0.2, 0.28):
            q = fit_quad(plane_quad(w, h, OUT_W / 2, OUT_H * cy, random.choice([-14, 14]), -8, scale))
            poly = np.zeros((OUT_H // 4, OUT_W // 4), np.uint8); cv2.fillConvexPoly(poly, (q / 4).astype(np.int32), 1)
            vis = bg[::4, ::4][poly > 0].mean() if poly.any() else 0
            if best is None or vis > best[0]: best = (vis, q)
        self.vis, self.quad0 = best
        self.behind = self.vis >= 0.55 and mask.mean() > 0.03   # si casi no se ve el fondo, va delante
        self.H = np.eye(3); self.prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        self.pts = cv2.goodFeaturesToTrack(self.prev, 300, 0.01, 12, mask=((mask < 0.5) * 255).astype(np.uint8))
    def update(self, frame):
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if self.pts is not None and len(self.pts) >= 8:
            nxt, st, _ = cv2.calcOpticalFlowPyrLK(self.prev, gray, self.pts, None, winSize=(21, 21), maxLevel=3)
            g0, g1 = self.pts[st[:, 0] == 1], nxt[st[:, 0] == 1]
            if len(g0) >= 8:
                Hs, _ = cv2.findHomography(g0, g1, cv2.RANSAC, 3.0)
                if Hs is not None and abs(np.linalg.det(Hs[:2, :2]) - 1) < 0.2:
                    Hn = Hs @ self.H; c0 = self.quad0.mean(0); c1 = cv2.perspectiveTransform(c0.reshape(1, 1, 2), Hn)[0, 0]
                    if np.abs(c1 - c0).max() < OUT_W * 0.2: self.H = Hn
                self.pts = g1.reshape(-1, 1, 2)
        if self.pts is None or len(self.pts) < 60: self.pts = cv2.goodFeaturesToTrack(gray, 300, 0.01, 12)
        self.prev = gray
    def draw(self, frame, mask, t, T):
        q = fit_quad(cv2.perspectiveTransform(self.quad0.reshape(-1, 1, 2), self.H).reshape(4, 2))
        c = q.mean(0); rise = (1 - min(1, t / 0.5)) ** 2; q = (q - c) * (0.85 + 0.15 * (1 - rise)) + c + np.array([0, 40 * rise])
        h, w = self.rgba.shape[:2]
        M = cv2.getPerspectiveTransform(np.float32([[0, 0], [w, 0], [w, h], [0, h]]), q.astype(np.float32))
        wp = cv2.warpPerspective(self.rgba, M, (OUT_W, OUT_H), flags=cv2.INTER_LINEAR, borderValue=(0, 0, 0, 0))
        a = wp[..., 3:4].astype(np.float32) / 255 * min(1, t / 0.35, (T - t) / 0.3)
        if self.behind: a *= (1 - mask)[..., None]
        frame[:] = (frame * (1 - a) + wp[..., [2, 1, 0]] * a).astype(np.uint8)

# ---------- B-roll: pantalla completa, polaroid y TV vieja ----------
def blur_fill(img):
    bg = crop_frame(img, 0.5, 1.0); bg = cv2.GaussianBlur(cv2.resize(bg, (OUT_W // 6, OUT_H // 6)), (0, 0), 6)
    return (cv2.resize(bg, (OUT_W, OUT_H)) * 0.55).astype(np.uint8)
def fit_into(img, W, H):
    h, w = img.shape[:2]; k = min(W / w, H / h); return cv2.resize(img, (int(w * k), int(h * k)), interpolation=cv2.INTER_AREA)
class BrollPlayer:
    def __init__(self, ev):
        self.ev = ev; b = ev['b']; self.cap = None
        if b['kind'] == 'video':
            self.cap = cv2.VideoCapture(b['proxy']); n = b.get('nframes', 0)
            span = ev['f1'] - ev['f0']
            if n > span + 10: self.cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(0, n - span - 1))
        self.last = b.get('img'); self.rot = random.uniform(-4, 4); self.counter = random.randint(100, 900)
    def frame(self):
        if self.cap is not None:
            ok, fr = self.cap.read()
            if ok: self.last = fr
        return self.last if self.last is not None else np.zeros((720, 1280, 3), np.uint8)
    def draw(self, out, g):
        src = self.frame(); t = (g - self.ev['f0']) / FPS; T = (self.ev['f1'] - self.ev['f0']) / FPS
        style = self.ev['style']; z = 1 + 0.04 * t / T
        out[:] = blur_fill(src)
        if style == 'pantalla':
            if src.shape[1] / src.shape[0] < 0.75: out[:] = crop_frame(src, 0.5, z)
            else:
                im = fit_into(src, int(OUT_W * 0.96 * z), int(OUT_H * 0.6)); h, w = im.shape[:2]
                out[(OUT_H - h) // 2:(OUT_H - h) // 2 + h, (OUT_W - w) // 2:(OUT_W - w) // 2 + w] = im
        elif style == 'polaroid':
            out[:] = (out * 0.35 + GRUNGE[1] * 0.65).astype(np.uint8)
            im = fit_into(src, int(OUT_W * 0.74), int(OUT_H * 0.46)); h, w = im.shape[:2]
            m_, bot = 26, 120
            card = np.full((h + m_ + bot, w + 2 * m_, 4), (238, 236, 230, 255), np.uint8)
            card[m_:m_ + h, m_:m_ + w, :3] = im[..., ::-1]
            ang = self.rot + 1.5 * math.sin(t * 1.3); s_ = 0.92 + 0.08 * min(1, t / 0.3)
            ch, cw = card.shape[:2]; M = cv2.getRotationMatrix2D((cw / 2, ch / 2), ang, s_)
            cos, sin = abs(M[0, 0]), abs(M[0, 1]); nw, nh = int(ch * sin + cw * cos), int(ch * cos + cw * sin)
            M[0, 2] += nw / 2 - cw / 2; M[1, 2] += nh / 2 - ch / 2
            rc = cv2.warpAffine(card, M, (nw, nh), borderValue=(0, 0, 0, 0))
            sh = rc.copy(); sh[..., :3] = 0; sh[..., 3] = cv2.GaussianBlur(sh[..., 3], (0, 0), 18) // 2
            x0, y0 = (OUT_W - nw) // 2, int(OUT_H * 0.42) - nh // 2
            paste_rgba(out, sh, x0 + 18, y0 + 26); paste_rgba(out, rc, x0, y0)
        else:   # tv vieja
            sw, sh_ = int(OUT_W * 0.86), int(OUT_W * 0.86 * 0.78)
            scr = crop_frame(src, 0.5, z, sw, sh_)
            scr = cv2.addWeighted(scr, 0.9, cv2.GaussianBlur(scr, (0, 0), 2), 0.25, 0)
            scr[::4] = (scr[::4] * 0.72).astype(np.uint8)                           # líneas de barrido
            yy, xx = np.mgrid[0:sh_, 0:sw]; v = 1 - 0.45 * (((xx - sw / 2) / (sw / 2)) ** 2 + ((yy - sh_ / 2) / (sh_ / 2)) ** 2) ** 1.5
            scr = (scr * np.clip(v, 0, 1)[..., None]).astype(np.uint8)
            b_, g_, r_ = cv2.split(scr); scr = cv2.merge([np.roll(b_, -2, 1), g_, np.roll(r_, 2, 1)])
            f = font('VT323-Regular.ttf', 54); pil = Image.fromarray(scr[..., ::-1]); d = ImageDraw.Draw(pil)
            d.text((sw / 2 - 60, sh_ - 70), f"{self.counter + g - self.ev['f0']:06d}", font=f, fill=(225, 225, 225))
            scr = np.array(pil)[..., ::-1]
            out[:] = (out * 0.5).astype(np.uint8)
            x0, y0 = (OUT_W - sw) // 2, (OUT_H - sh_) // 2 - 80
            cv2.rectangle(out, (x0 - 40, y0 - 40), (x0 + sw + 40, y0 + sh_ + 40), (22, 22, 24), -1, cv2.LINE_AA)
            cv2.rectangle(out, (x0 - 40, y0 - 40), (x0 + sw + 40, y0 + sh_ + 40), (55, 55, 58), 4, cv2.LINE_AA)
            mask_ = np.zeros((sh_, sw), np.uint8); cv2.rectangle(mask_, (0, 0), (sw - 1, sh_ - 1), 255, -1)
            rr = 46; mask_[:rr, :rr] = 0; mask_[:rr, -rr:] = 0; mask_[-rr:, :rr] = 0; mask_[-rr:, -rr:] = 0
            for cx_, cy_ in ((rr, rr), (sw - rr, rr), (rr, sh_ - rr), (sw - rr, sh_ - rr)): cv2.circle(mask_, (cx_, cy_), rr, 255, -1)
            a = (mask_ / 255.0)[..., None]
            out[y0:y0 + sh_, x0:x0 + sw] = (out[y0:y0 + sh_, x0:x0 + sw] * (1 - a) + scr * a).astype(np.uint8)
            if random.random() < 0.08: out[y0:y0 + sh_, x0:x0 + sw] = cv2.add(out[y0:y0 + sh_, x0:x0 + sw], 25)   # parpadeo

# ---------- Trazos de luz neón ----------
class Neon:
    def __init__(self, ev):
        rng = np.random.default_rng(ev['f0']); n = 7
        pts = np.stack([np.linspace(-0.05, 1.05, n) * OUT_W, (0.35 + 0.25 * rng.random(n)) * OUT_H], 1)
        pts[:, 1] += np.sin(np.arange(n) * 2.1) * OUT_H * 0.05
        tt = np.linspace(0, 1, 400); idx = tt * (n - 1); i0 = np.clip(idx.astype(int), 0, n - 2); fr_ = idx - i0
        base = pts[i0] * (1 - fr_[:, None]) + pts[i0 + 1] * fr_[:, None]
        wig = np.stack([np.sin(tt * 38) * 55, np.cos(tt * 31) * 70], 1) * np.sin(tt * np.pi)[:, None]   # garabato
        self.path = (base + wig).astype(np.int32)
        self.col = [(255, 140, 40), (255, 60, 200), (60, 220, 255)][int(rng.integers(3))]
        self.ev = ev
    def draw(self, frame, g):
        t = (g - self.ev['f0']) / (self.ev['f1'] - self.ev['f0'])
        head = int(len(self.path) * min(1, t * 1.6)); tail = int(len(self.path) * max(0, t * 1.6 - 0.7))
        if head - tail < 2: return
        layer = np.zeros_like(frame); seg = self.path[tail:head].reshape(-1, 1, 2)
        cv2.polylines(layer, [seg], False, self.col[::-1], 34, cv2.LINE_AA)
        layer = cv2.GaussianBlur(layer, (0, 0), 16)
        cv2.polylines(layer, [seg], False, self.col[::-1], 12, cv2.LINE_AA)
        cv2.polylines(layer, [seg], False, (255, 255, 255), 5, cv2.LINE_AA)
        frame[:] = cv2.add(frame, layer)

# ---------- Spider-Verse: cómic (halftone, tinta, colores desfasados, animación a saltos) ----------
yy_, xx_ = np.mgrid[0:OUT_H, 0:OUT_W].astype(np.float32)
_ang = np.radians(18)
_u = (xx_ * np.cos(_ang) + yy_ * np.sin(_ang)) / 11; _v = (-xx_ * np.sin(_ang) + yy_ * np.cos(_ang)) / 11
DOTS = (np.sqrt((_u - np.round(_u)) ** 2 + (_v - np.round(_v)) ** 2) * 2).astype(np.float32)   # 0 en el centro del punto
def spiderverse(frame, t):
    f = frame.astype(np.float32)
    lum = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    f = np.round(f / 48) * 48                                                    # colores planos
    dots = (DOTS < (1 - lum) * 1.25).astype(np.float32)                           # puntos de imprenta en las sombras
    f *= (1 - 0.35 * dots)[..., None]
    b_, g_, r_ = cv2.split(f); off = 7 + int(3 * math.sin(t * 20))
    f = cv2.merge([np.roll(b_, -off, 1), g_, np.roll(np.roll(r_, off, 1), off // 2, 0)])   # desfase de impresión
    edges = cv2.Canny(cv2.GaussianBlur(frame, (5, 5), 0), 60, 140)
    edges = cv2.dilate(edges, np.ones((3, 3), np.uint8)).astype(np.float32)[..., None] / 255
    f = f * (1 - 0.85 * edges)                                                    # líneas de tinta
    return np.clip(f, 0, 255).astype(np.uint8)
print("Look listo.")

In [ ]:
#@title 8 · Render final 9:16 (1080×1920) para TikTok / Reels / Shorts
OUT = os.path.join(WORK, f"{re.sub(r'[^A-Za-z0-9]+', '_', NOMBRE_SERVER).strip('_').lower()}_promo_tiktok.mp4")
cmd = ['ffmpeg', '-y', '-v', 'error', '-f', 'rawvideo', '-pix_fmt', 'bgr24', '-s', f'{OUT_W}x{OUT_H}', '-r', str(FPS),
       '-i', '-', '-i', MIX_WAV, '-map', '0:v', '-map', '1:a', *VENC, '-pix_fmt', 'yuv420p',
       '-color_primaries', 'bt709', '-color_trc', 'bt709', '-colorspace', 'bt709',
       '-c:a', 'aac', '-b:a', '256k', '-t', f"{L:.4f}", '-movflags', '+faststart', OUT]
pipe = subprocess.Popen(cmd, stdin=subprocess.PIPE, stderr=subprocess.PIPE, cwd=WORK)
def active(lst, g): return next((x for x in lst if x['f0'] <= g < x['f1']), None)
t0 = time.time(); g = 0; scene_txt = {}; brolls = {}; neons = {}; sv_hold = None; cta_rgba = None
for pc in PIECES:
    c = clips[pc['clip']]; n = pc['n']
    cap = cv2.VideoCapture(c['proxy']); cap.set(cv2.CAP_PROP_POS_FRAMES, pc['src_f0']); last = None
    for i in range(n):
        ok, fr = cap.read()
        fr = fr if ok else (last if last is not None else np.zeros((c['H'], c['W'], 3), np.uint8)); last = fr
        tt = g / FPS
        card, br, fx = active(CARDS, g), active(BROLLS, g), active(FX, g)
        if card is not None:
            f = np.empty((OUT_H, OUT_W, 3), np.uint8); draw_card(f, card, g)
        elif br is not None:
            f = np.empty((OUT_H, OUT_W, 3), np.uint8)
            brolls.setdefault(id(br), BrollPlayer(br)).draw(f, g); f = grade(f)
        else:
            seg = max(j for j, m_ in enumerate(pc['zmarks']) if m_ <= i)
            s_start = pc['zmarks'][seg]; s_end = pc['zmarks'][seg + 1] if seg + 1 < len(pc['zmarks']) else n
            z = pc['zlevels'][seg] * (1 + 0.03 * (i - s_start) / max(1, s_end - s_start))
            cx = float(np.clip(pc['cx'][min(i, len(pc['cx']) - 1)] + pc['xoff'][seg], 0, 1))
            f = grade(crop_frame(fr, cx, z))
            if fx is not None and fx['kind'] == 'spiderverse':
                if (g - fx['f0']) % 2 == 1 and sv_hold is not None: f = sv_hold.copy()   # "animación a dos" (12 fps)
                else: f = spiderverse(f, tt); sv_hold = f.copy()
            ttl = active(TITLES, g)
            if ttl is not None:
                mask = person_mask(f)
                if id(ttl) not in scene_txt: scene_txt[id(ttl)] = SceneText(ttl, f, mask)
                else: scene_txt[id(ttl)].update(f)
                scene_txt[id(ttl)].draw(f, mask, (g - ttl['f0']) / FPS, (ttl['f1'] - ttl['f0']) / FPS)
        if fx is not None and fx['kind'] == 'neon' and card is None:
            neons.setdefault(id(fx), Neon(fx)).draw(f, g)
        if g >= CTA['f0'] and TITULOS_3D_DORADOS:          # cierre: llamada a la acción en dorado
            if cta_rgba is None:
                cta_rgba = gold3d_rgba(CTA['text'], 120)
                if cta_rgba.shape[1] > OUT_W * 0.9: cta_rgba = cv2.resize(cta_rgba, None, fx=OUT_W * 0.9 / cta_rgba.shape[1], fy=OUT_W * 0.9 / cta_rgba.shape[1])
            k_ = (g - CTA['f0']) / FPS
            paste_rgba(f, cta_rgba, (OUT_W - cta_rgba.shape[1]) // 2, int(OUT_H * 0.12), min(1, k_ / 0.4))
        if card is None: draw_caption(f, tt)
        f = finish(f)
        try:
            pipe.stdin.write(f.tobytes())
        except BrokenPipeError:
            raise RuntimeError("ffmpeg se detuvo: " + pipe.stderr.read().decode()[-2000:])
        g += 1
    cap.release()
    print(f"\r  {g}/{TOTAL_FRAMES} frames ({g / max(1e-6, time.time() - t0):.1f} fps)", end='')
pipe.stdin.close(); err = pipe.stderr.read().decode(); pipe.wait()
if pipe.returncode != 0:
    raise RuntimeError(err[-2000:])
print(f"\n✓ {OUT}")

In [ ]:
#@title 9 · Guardar en Drive (carpeta Edicion_IA) + guion y notas
notes = [f"# {NOMBRE_SERVER} — promo TikTok", "", PLAN.get('resumen', ''), "",
         f"- Duración: {L:.1f}s · {len(PIECES)} planos · gancho: **{PLAN['gancho']}** · CTA: **{PLAN['cta']}**", "",
         "| # | Clip | Lo que dices | En grande | Pantalla negra | Capítulo 3D | B-roll | Efecto | Canción |",
         "|---|---|---|---|---|---|---|---|---|"]
for k, g_ in enumerate(PLAN['guion']):
    p = byid[g_['id']]
    song = f"{g_['pedido_cancion']} ({g_['segundos_cancion']:.0f}s{', ' + g_['cancion'] if g_['cancion'] else ', en TikTok'})" if g_['pedido_cancion'] else '—'
    br = f"{g_['broll']} ({g_['estilo_broll']})" if g_['broll'] else '—'
    notes.append(f"| {k + 1} | {clips[p['clip']]['name']} @ {p['start']:.1f}s | {p['text']} | {g_['enfasis'] or '—'} | "
                 f"{g_['tarjeta'] or '—'} | {g_['capitulo'] or '—'} | {br} | {g_['efecto'] or '—'} | {song} |")
if CREDITO_MUSICA.strip():
    notes += ["", f"**Créditos:** {CREDITO_MUSICA.strip()}"]
open(f"{WORK}/guion_promo_tiktok.md", 'w', encoding='utf-8').write("\n".join(notes))

# Lista para poner las canciones desde "Agregar sonido" en TikTok
mmss = lambda x: f"{int(x // 60)}:{int(x % 60):02d}"
lst = [f"Canciones para poner en TikTok — {NOMBRE_SERVER}", "",
       "En TikTok: Agregar sonido → busca la canción → recórtala → colócala en el minuto indicado.", ""]
for e in sorted(SONG_MARKS + SONG_EVENTS, key=lambda e: e['t0']):
    where = "ya va dentro del video" if e['file'] else "ponla desde TikTok"
    lst.append(f"{mmss(e['t0'])}  →  «{e['name']}»  ({e['dur']:.0f} s, {where})")
if len(lst) == 4: lst.append("(no se detectaron pedidos de canción en el video)")
open(f"{WORK}/canciones_para_tiktok.txt", 'w', encoding='utf-8').write("\n".join(lst) + "\n")
print("\n".join(lst[4:]))
for p in (OUT, f"{WORK}/guion_promo_tiktok.md", f"{WORK}/transcripcion.txt", f"{WORK}/canciones_para_tiktok.txt"):
    shutil.copy(p, OUT_DIR); print("→", os.path.join(OUT_DIR, os.path.basename(p)))
if IN_COLAB:
    from IPython.display import Video, display
    prev = f"{WORK}/preview.mp4"
    run(['ffmpeg', '-y', '-v', 'error', '-i', OUT, '-vf', 'scale=360:-2', '-c:v', 'libx264', '-crf', '30', '-c:a', 'aac', '-b:a', '96k', prev])
    display(Video(prev, embed=True, width=300))
print("\n¿Ajustes? Edita PLAN['guion'] ('enfasis', 'tarjeta', 'broll', 'estilo_broll', 'capitulo', 'efecto'), PLAN['gancho'] / PLAN['cta'] y vuelve a correr las celdas 6–9.")